In [23]:
import pandas as pd
from datetime import date
import discogs_client
import requests
import json
import time

In [41]:
df = pd.read_csv("/workspaces/Personal_Coding/Discogs Collection Modifier/zackdimondrecords1-collection-20260403-1317.csv")


In [42]:
backup = df


In [43]:
df_stock = df.dropna(subset =['Collection Notes'])
df_stock

,Catalog#,Artist,Title,Label,Format,Rating,Released,release_id,CollectionFolder,Date Added,Collection Price,Collection Sold Price,Collection Notes
0,"S 4208, 4208",Jerry Goldsmith,Patton (Original Motion Picture Score),"20th Century Fox Records, 20th Century Fox Rec...","LP, Album",NaN,1970,29799220,Record Collection #100,2025-07-28 11:17:56,NaN,NaN,A
3,4S-SP-106,Various,Made In Tennessee,4 Star Records,"LP, Comp",NaN,1977,4454379,Record Collection #97,2025-06-24 12:37:49,NaN,NaN,A
5,SP 5199,Amy Grant,Lead Me On,A&M Records,"LP, Album",NaN,1988,2096172,Record Collection #111,2026-01-23 14:42:02,7,NaN,A
7,SP-4249,Lee Michaels,Barrel,A&M Records,"LP, Album",NaN,1970,1226289,Record Collection #98,2026-04-01 22:36:08,NaN,NaN,A
8,SP-4527,Joan Baez,Diamonds & Rust,A&M Records,"LP, Album, Pit",NaN,1975,7586003,Record Collection #117,2026-04-02 15:04:18,5,NaN,A
...,...,...,...,...,...,...,...,...,...,...,...,...,...
689,WST-8490-LP,Dr. Charles S. Kendall,Morning Chimes,Word,"LP, Album",NaN,1970,5356478,Record Collection #97,2025-06-24 11:06:50,NaN,NaN,A
690,WST-8714,Evie (2),Gentle Moments,Word,LP,NaN,1976,3195892,Record Collection #111,2026-01-23 14:34:01,NaN,NaN,A
691,WST-8770,Evie (2),"Come On, Ring Those Bells",Word,"LP, Album, Lar",NaN,1977,17289814,Record Collection #111,2026-01-25 15:15:07,NaN,NaN,A
692,"WSB-8769, WSB 8769",Evie (2),A Little Song Of Joy For My Little Friends,"Word, Word","LP, Album",NaN,1978,1451769,Record Collection #111,2026-01-23 15:02:57,NaN,NaN,A


In [44]:
df.columns

Index(['Catalog#', 'Artist', 'Title', 'Label', 'Format', 'Rating', 'Released',
       'release_id', 'CollectionFolder', 'Date Added', 'Collection Price',
       'Collection Sold Price', 'Collection Notes'],
      dtype='str')

In [45]:
df_sold = df[df['Collection Notes'].isna()]
df_sold.head()

,Catalog#,Artist,Title,Label,Format,Rating,Released,release_id,CollectionFolder,Date Added,Collection Price,Collection Sold Price,Collection Notes
1,T-414,Love Unlimited,Under The Influence Of Love Unlimited,20th Century Records,"LP, Album, P/Mixed, Ter",NaN,1973,580795,Record Collection #111,2026-01-25 18:22:01,NaN,NaN,NaN
2,T-433,Love Unlimited Orchestra,Rhapsody In White,20th Century Records,"LP, Album",NaN,1974,21270940,Individual Find,2026-03-23 07:07:24,NaN,NaN,NaN
4,SE 1024,Carpenters,Yesterday Once More,A&M Records,"2xLP, Comp, Mon",NaN,1984,3110567,Record Collection #106,2025-10-24 14:32:37,NaN,NaN,NaN
6,SP-3708,Supertramp,Breakfast In America,A&M Records,"LP, Album, Mon",NaN,1979,4450663,Record Collection #116,2026-03-11 11:51:35,NaN,NaN,NaN
11,SP-4896,Jeffrey Osborne,Jeffrey Osborne,A&M Records,"LP, Album, R,",NaN,1982,1545418,Record Collection #89,2025-05-07 07:34:34,NaN,NaN,NaN


In [46]:
df_sold['Blank'] = ''
df_sold =df_sold[['Blank','Artist', 'Title', 'CollectionFolder','Collection Price','Collection Sold Price','Date Added']]
df_sold.head()

,Blank,Artist,Title,CollectionFolder,Collection Price,Collection Sold Price,Date Added
1,,Love Unlimited,Under The Influence Of Love Unlimited,Record Collection #111,NaN,NaN,2026-01-25 18:22:01
2,,Love Unlimited Orchestra,Rhapsody In White,Individual Find,NaN,NaN,2026-03-23 07:07:24
4,,Carpenters,Yesterday Once More,Record Collection #106,NaN,NaN,2025-10-24 14:32:37
6,,Supertramp,Breakfast In America,Record Collection #116,NaN,NaN,2026-03-11 11:51:35
11,,Jeffrey Osborne,Jeffrey Osborne,Record Collection #89,NaN,NaN,2025-05-07 07:34:34


In [47]:
df_sold['CollectionFolder'].value_counts()


CollectionFolder
Record Collection #106    57
Record Collection #111    54
Record Collection #98     19
Record Collection #105    19
Individual Find           15
Record Collection #89     12
Record Collection #109    12
Record Collection #113    10
Record Collection #102     8
Record Collection #96      8
Record Collection #93      7
Record Collection #103     7
Uncategorized              6
Record Collection #104     5
Record Collection #100     5
Record Collection #114     5
Record Collection #97      5
Record Collection #108     4
Record Collection #112     4
Record Collection #101     3
Record Collection #91      3
Record Collection #107     2
Record Collection #116     1
Record Collection #117     1
Record Collection #92      1
Record Collection #115     1
Name: count, dtype: int64

In [48]:
df_sold['CollectionFolder']=df_sold['CollectionFolder'].replace('Uncategorized', 'Individual Find')

In [49]:
today = pd.Timestamp.today().normalize()

In [50]:
df_sold['Date Added'] = pd.to_datetime(df_sold['Date Added'])
df_sold.loc[
    df_sold['CollectionFolder'] == 'Individual Find',
    'CollectionFolder'
] = (today - df_sold['Date Added']).dt.days.astype(str)



In [51]:
df_sold['Collection Price'] = df_sold['Collection Price'].fillna(3)
df_sold['Collection Sold Price'] = df_sold['Collection Sold Price'].fillna(df_sold['Collection Price'])
df_sold['Collection Sold Price'].value_counts()

Collection Sold Price
3       272
22.5      1
5         1
Name: count, dtype: int64

In [52]:
df_sold=df_sold[['Blank','Artist', 'Title', 'CollectionFolder','Collection Price','Collection Sold Price']]
df_sold.head()

,Blank,Artist,Title,CollectionFolder,Collection Price,Collection Sold Price
1,,Love Unlimited,Under The Influence Of Love Unlimited,Record Collection #111,3,3
2,,Love Unlimited Orchestra,Rhapsody In White,14,3,3
4,,Carpenters,Yesterday Once More,Record Collection #106,3,3
6,,Supertramp,Breakfast In America,Record Collection #116,3,3
11,,Jeffrey Osborne,Jeffrey Osborne,Record Collection #89,3,3


In [53]:
df_sold.to_csv("Sold.csv", index=False)

In [73]:
df_stock.columns

Index(['Catalog#', 'Artist', 'Title', 'Label', 'Format', 'Rating', 'Released',
       'release_id', 'CollectionFolder', 'Date Added', 'Collection Price',
       'Collection Sold Price', 'Collection Notes'],
      dtype='str')

In [83]:
df_stock= df_stock[['Artist', 'Title', 'Collection Price']]
df_stock['Collection Price']= df_stock['Collection Price'].fillna(3)

In [63]:
#getting the info of the page

#defining the username
username='ZackDimondRecords1'

#defining the folder_id
folder_id=1
#connecting discogs to this page
headers = {
    #myuserID
    "User-Agent": "Inventory_Remover_ZDR",
    #mytoken
    "Authorization": "Discogs token=ONQcVBLePsQvGKasmFsIdIdEsKmkvOxKVwntWgdZ"
}
#setting the page count
page = 1
#setting how many items appear in the list
per_page = 50
#setting the folder id
folder_id = 0
#define the collection
collection_url = f"https://api.discogs.com/users/{username}/collection/folders/{folder_id}/releases?page={page}&per_page={per_page}"
#make request
response = requests.get(collection_url, headers=headers)
data = response.json()
with open("output.json", "w") as f:
    json.dump(data, f, indent=4)

In [64]:
pages_n = response.json()['pagination']['pages']

var_page = 0
delete_list = []
true_counter = 0


while var_page < pages_n:
    var_page += 1

    var_url = f"https://api.discogs.com/users/{username}/collection/folders/{folder_id}/releases?page={var_page}&per_page={per_page}"
    var_response = requests.get(var_url, headers=headers)

    if var_response.status_code != 200:
        print(f"Error {var_response.status_code}")
        continue

    var_json = var_response.json()
    releases = var_json.get('releases', [])

    for release in releases:

        notes = release.get("notes") or []

        # count "A" safely (optional, fixed version)
        if len(notes) > 0 and notes[0].get("value") == "A":
            true_counter += 1

        # MAIN RULE: field_id == 5
        if any(n.get("field_id") == 5 for n in notes):
            delete_list.append({
                "folder_id": release["folder_id"],
                "release_id": release["id"],
                "instance_id": release["instance_id"]
            })

print(delete_list)

[{'folder_id': 9153430, 'release_id': 27276714, 'instance_id': 2108359124}, {'folder_id': 1, 'release_id': 422477, 'instance_id': 2132362106}]


In [65]:
for item in delete_list:
    url = f"https://api.discogs.com/users/{username}/collection/folders/{item['folder_id']}/releases/{item['release_id']}/instances/{item['instance_id']}"

    response = requests.delete(url, headers=headers)

    if response.status_code == 204:
        print("✅ Deleted")
    elif response.status_code == 429:
        print("⏳ Rate limited — sleeping...")
        time.sleep(10)  # wait longer if blocked
        continue
    else:
        print("❌ Error:", response.status_code, response.text)

    time.sleep(1.5)  # normal throttle


✅ Deleted
✅ Deleted
